# Introduction

bla bla bla

# Preparation and extraction of data

In [22]:
import pandas as pd
import alive_progress

In [23]:
FILE_PATHS = [
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2024.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2023.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2022.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2021.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2020.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2019.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2018.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2017.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2016.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2015.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2014.xlsx',
]

As the memory taken by the dataset is sufficiently low, there is no need to actually chunk it down in order to process it and I can simply load the datasets in memory and concatenate them

Now, we can save the dataset corresponding only to university students

In [24]:
completeDF = pd.DataFrame()

with alive_progress.alive_bar(len(FILE_PATHS), force_tty=True) as bar:
    for dataset in FILE_PATHS:
        df = pd.read_excel(dataset)
        # We are only interested in partecipants who are exchanging students in different universities, as such we will remove all other partecipants from the dataset
        filteredDF = df[df["Activity (mob)"].isin(['HE-LM-SMS - Student mobility for studies', 
                                                   'HE-SMS - Student mobility for studies', 
                                                   'HE-SMT - Student mobility for traineeships', 
                                                   'HE-LM-SMT - Student mobility for traineeships', 
                                                   'Student mobility for studies between Programme Countries',
                                                   'Student mobility for traineeships between Programme Countries'
                                                ])]
        
        completeDF = pd.concat([completeDF, filteredDF])
        # Save some memory
        del df, filteredDF
        #Move the animated progress bar
        bar()

completeDF.memory_usage(index=True).sum()

|████████████████████████████████████████| 11/11 [100%] in 5:53.4 (0.03/s)      ▄ 0/11 [0%] in 44s (~0s, 0.0/s)  ▃▁▃ 0/11 [0%] in 45s (~0s, 0.0/s)  ▄▆█ 0/11 [0%] in 46s (~0s, 0.0/s) 


np.int64(685362816)

In [26]:
print(completeDF.columns)

Index(['Academic Year', 'Mobility Start Month', 'Mobility Duration',
       'Activity (mob)', 'Field', 'Field of Education', 'Participant Country',
       'Education Level', 'Participant Gender', 'Participant Profile',
       'Fewer Opportunities', 'Participant Age', 'Sending Country',
       'Sending City', 'Sending Organization', 'Receiving Country',
       'Receiving City', 'Receiving Organization',
       'Actual Participants (Contracted Projects)', 'Project Reference',
       'Mobility Start Year/Month', 'Actual Participants',
       'Actual Participants ', 'Mobility Duration - calendar days',
       'Sending Organisation', 'Receiving Organisation'],
      dtype='str')


As our work focuses on university students, we can safely drop some columns:
- Moblity  Start Month: Is it of any use to our analysis? 
- Activity : As it has already been used for filtering
- Field : As it is no longer useful after the filtering has been completed
- Partecipant Profile: All of these will be Learners once we are done with the filtering
- Age: It is one of the least populated columns, with plenty of nulls, but might be useful so I will keep it
- Actual Partecipants is the last column of most datasets, I don't really know wheter it can be useful for us
-      'Project Reference',
       'Mobility Start Year/Month', 'Actual Participants',
       'Actual Participants ', 'Mobility Duration - calendar days',
       'Sending Organisation', 'Receiving Organisation', 'Project Reference'
are all column I...quite frankly do not know where they come from, I've checked all Datasets but none of them contain such columns, and all I did was stack them with concat. In any case, let's trim them and check later if the result is coherent. They are shown to be from 2015, but I have not found them in the 2015 Dataset

In [27]:
completeDF = completeDF.drop([
    'Mobility Start Month', 
    'Activity (mob)',
    'Field',
    'Participant Profile',
    'Project Reference',
    'Mobility Start Year/Month',
    'Actual Participants',
    'Actual Participants ',
    'Mobility Duration - calendar days',
    'Sending Organisation',
    'Receiving Organisation',
    'Project Reference'], axis=1)

In [28]:
print(completeDF.shape)
completeDF.head()

(3172976, 15)


,Academic Year,Mobility Duration,Field of Education,Participant Country,Education Level,Participant Gender,Fewer Opportunities,Participant Age,Sending Country,Sending City,Sending Organization,Receiving Country,Receiving City,Receiving Organization,Actual Participants (Contracted Projects)
1358,2023-24,61.0,"0410 - Business and administration, not furthe...",Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,0,22,AT - Austria,WIEN,WU,DE - Germany,Munich,-,1.0
1359,2023-24,101.0,0311 - Economics,Germany,ISCED-7 - Second cycle / Master’s or equivalen...,Male,0,27,AT - Austria,WIEN,WU,FR - France,Toulouse,Université Toulouse Capitole,1.0
1360,2023-24,121.0,"0410 - Business and administration, not furthe...",Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,0,22,AT - Austria,WIEN,WU,FR - France,Paris,-,1.0
1361,2023-24,171.0,"0410 - Business and administration, not furthe...",Germany,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,0,22,AT - Austria,WIEN,WU,FR - France,Paris,-,1.0
1365,2023-24,86.0,"0410 - Business and administration, not furthe...",Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,0,23,AT - Austria,WIEN,WU,DE - Germany,Hamburg,-,1.0


In [29]:
completeDF = completeDF.drop_duplicates()
print(completeDF.shape)

(3152472, 15)


In [31]:
print(completeDF.isnull().sum())

Academic Year                                      0
Mobility Duration                             152125
Field of Education                                 0
Participant Country                                0
Education Level                                    0
Participant Gender                                 0
Fewer Opportunities                                0
Participant Age                                    0
Sending Country                                    0
Sending City                                       0
Sending Organization                          152125
Receiving Country                                  0
Receiving City                                    12
Receiving Organization                        152131
Actual Participants (Contracted Projects)    1665496
dtype: int64


As over half of the values in Actual Participants is NaN, I'd drop the column. Mobility Duration, Sending Organization and Receiving Organization's NaN could be related, look into the Dataset

In [34]:
completeDF = completeDF.drop(['Actual Participants (Contracted Projects)'], axis=1)
completeDF = completeDF.dropna()
completeDF.shape

(3000330, 14)

In [36]:
completeDF.to_csv('Datasets/Processed/Erasmus-Data.csv')